# NeuroCodec — ConvLSTM Decoder Training

**Copyright (c) 2026 Chandini Narayana — MIT License**

Train the ConvLSTM video reconstructor on Google Colab (free T4 GPU).

**Runtime:** ~3 hours for 50 epochs on DAVIS240C  
**Output:** `decoder_v1.pt` saved to Google Drive

## Steps
1. Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Download `decoder_v1.pt` from Google Drive
4. Place it at `neurocodec/weights/decoder_v1.pt`

In [ ]:
# ── 0. Install dependencies ───────────────────────────────────────────────
!pip install -q torch torchvision opencv-python-headless scikit-image matplotlib
!pip install -q 'neurocodec @ git+https://github.com/chandini-narayana/neurocodec.git'

import torch
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/neurocodec'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')

In [ ]:
# ── 2. Download DAVIS240C dataset ────────────────────────────────────────
# Dataset: https://rpg.ifi.uzh.ch/davis_data.html
# License: CC-BY 4.0
# This cell downloads a small subset (~500MB) for quick experiments.
# For full training, download the complete dataset manually.

import os
DATA_DIR = '/content/davis240c'
os.makedirs(DATA_DIR, exist_ok=True)

# Download poster_translation sequence (good for testing)
!wget -q -O /content/poster_6dof.zip \
    'http://rpg.ifi.uzh.ch/docs/RAL18_Rebecq/poster_6dof.zip' \
    || echo 'Manual download required: https://rpg.ifi.uzh.ch/davis_data.html'

# Alternative: use your own dashcam footage
# Upload to /content/ via the Files panel on the left
print('Data directory:', os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'Empty — upload video files')

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────────
import torch

CFG = {
    # Data
    'video_dir':    DATA_DIR,
    'video_ext':    ('.mp4', '.avi', '.mov'),
    'img_h':        180,
    'img_w':        240,
    'seq_len':      8,          # frames per training sequence

    # Encoder params (must match at inference time)
    'threshold':    0.2,
    'n_bins':       5,
    'k_frac':       0.1,

    # Model
    'hidden_dims':  [64, 64, 32],

    # Training
    'lr':           1e-4,
    'batch_size':   4,
    'n_epochs':     50,
    'ckpt_every':   500,        # steps
    'loss_alpha':   0.5,        # L1 weight; (1-alpha) = perceptual

    # Paths
    'drive_dir':    DRIVE_DIR,
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
}
print('Config:', CFG)

In [ ]:
# ── 4. Dataset ───────────────────────────────────────────────────────────
import cv2, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from neurocodec.simulator import DVSSimulator
from neurocodec.encoder import VoxelGrid

class VideoEventDataset(Dataset):
    """
    Streams (voxel_sequence, frame_sequence) pairs from video files.
    Each item: voxel [seq_len, B, H, W], target frames [seq_len, 1, H, W].
    """
    def __init__(self, video_dir, cfg):
        self.cfg  = cfg
        self.clips = []
        for ext in cfg['video_ext']:
            self.clips += list(Path(video_dir).rglob(f'*{ext}'))
        if not self.clips:
            # Fallback: generate synthetic moving-gradient video
            print('No video files found - using synthetic gradient sequence (smoke test only!)')
            self.synthetic = True
        else:
            self.synthetic = False
            print(f'Found {len(self.clips)} video file(s)')

        self.sim = DVSSimulator(threshold=cfg['threshold'])
        self.vg  = VoxelGrid(n_bins=cfg['n_bins'])
        self.seq_len = cfg['seq_len']
        self.H, self.W = cfg['img_h'], cfg['img_w']

        # Pre-index: list of (clip_idx, start_frame)
        self._index = self._build_index()

    def _build_index(self):
        if self.synthetic:
            return [(None, i) for i in range(200)]
        index = []
        for ci, clip in enumerate(self.clips):
            cap = cv2.VideoCapture(str(clip))
            n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            for s in range(0, n - self.seq_len - 1, self.seq_len):
                index.append((ci, s))
        return index

    def __len__(self): return len(self._index)

    def _synthetic_frames(self, idx):
        """Generate synthetic moving gradient frames for testing."""
        H, W = self.H, self.W
        frames = []
        for t in range(self.seq_len + 1):
            shift = (idx * self.seq_len + t) % W
            x = np.linspace(0, 1, W)
            row = np.roll(x, shift)
            frame = np.tile(row, (H, 1)).astype(np.float32)
            # Add some vertical variation
            y = np.linspace(0, 0.3, H).reshape(-1, 1)
            frame = np.clip(frame + y + 0.1 * np.random.randn(H, W), 0, 1)
            bgr = (frame[:, :, None].repeat(3, axis=2) * 255).astype(np.uint8)
            frames.append(bgr)
        return np.array(frames)

    def __getitem__(self, idx):
        ci, start = self._index[idx]

        if self.synthetic:
            frames_bgr = self._synthetic_frames(idx)
        else:
            cap = cv2.VideoCapture(str(self.clips[ci]))
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            frames_bgr = []
            for _ in range(self.seq_len + 1):  # +1 for diff
                ret, f = cap.read()
                if not ret: break
                f = cv2.resize(f, (self.W, self.H))
                frames_bgr.append(f)
            cap.release()
            if len(frames_bgr) < self.seq_len + 1:
                # Pad with last frame
                while len(frames_bgr) < self.seq_len + 1:
                    frames_bgr.append(frames_bgr[-1])
            frames_bgr = np.array(frames_bgr)

        stream = self.sim.simulate_array(frames_bgr)
        grids  = self.vg.build(stream)  # (T, B, H, W)

        # Target: grayscale frames in [0, 1]
        targets = []
        for f in frames_bgr[1:self.seq_len + 1]:
            g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            targets.append(g[None])  # (1, H, W)

        # grids[i] holds the events up to frame i (events between frame i-1 and i),
        # so it is paired with frame i.  grids[0] has no events and is skipped.
        T = min(len(grids) - 1, self.seq_len, len(targets))
        voxels  = torch.from_numpy(grids[1:T + 1])          # (T, B, H, W)
        targets = torch.from_numpy(np.array(targets[:T]))   # (T, 1, H, W)
        return voxels, targets


from torch.utils.data import random_split

full_ds = VideoEventDataset(CFG['video_dir'], CFG)
n_val   = max(1, int(0.1 * len(full_ds)))
train_ds, val_ds = random_split(
    full_ds, [len(full_ds) - n_val, n_val],
    generator=torch.Generator().manual_seed(0))
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'],
                      shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds, batch_size=CFG['batch_size'],
                      shuffle=False, num_workers=2, pin_memory=True)
print(f'Dataset: {len(train_ds)} train / {len(val_ds)} val sequences | '
      f'Batches: {len(train_dl)}')

In [ ]:
# ── 5. Model + loss ──────────────────────────────────────────────────────
import torch.nn as nn
from neurocodec.decoder import ConvLSTMDecoder, CombinedLoss

device = torch.device(CFG['device'])
model  = ConvLSTMDecoder(n_bins=CFG['n_bins'], hidden_dims=CFG['hidden_dims']).to(device)
loss_fn = CombinedLoss(device=device, alpha=CFG['loss_alpha'])
optim   = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=CFG['n_epochs'])

print(f'Model params: {model.count_parameters():,}')
print(f'Device: {device}')

In [ ]:
# ── 6. Overfit test (1 batch) — verify architecture works ───────────────
print('Running 100-step overfit test on 1 batch...')
model.train()
voxels, targets = next(iter(train_dl))
voxels  = voxels.to(device)    # (B, T, C, H, W)
targets = targets.to(device)

for step in range(100):
    model.reset_state()
    B_batch, T, C, H, W = voxels.shape
    preds = []
    for t in range(T):
        pred = model(voxels[:, t])  # (B, 1, H, W)
        preds.append(pred)
    preds = torch.stack(preds, dim=1)  # (B, T, 1, H, W)

    loss = loss_fn(preds.view(-1, 1, H, W), targets.view(-1, 1, H, W))
    optim.zero_grad()
    loss.backward()
    optim.step()

    if step % 20 == 0:
        print(f'  Step {step:3d} | loss={loss.item():.4f}')

print('Overfit test complete. If loss is decreasing, architecture is correct.')

In [ ]:
# ── 7. Full training loop ────────────────────────────────────────────────
from pathlib import Path
import time

# Compute SSIM for validation
from skimage.metrics import structural_similarity as ssim_metric

def compute_ssim_batch(pred, target):
    """pred, target: (B, 1, H, W) tensors in [0,1]"""
    scores = []
    for i in range(pred.shape[0]):
        p = pred[i, 0].cpu().numpy()
        t = target[i, 0].cpu().numpy()
        scores.append(ssim_metric(p, t, data_range=1.0))
    return float(np.mean(scores))

CKPT_DIR = Path(CFG['drive_dir']) / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def evaluate(dl):
    model.eval()
    scores = []
    with torch.no_grad():
        for voxels, targets in dl:
            voxels, targets = voxels.to(device), targets.to(device)
            model.reset_state()
            preds = torch.stack([model(voxels[:, t]) for t in range(voxels.shape[1])], dim=1)
            scores.append(compute_ssim_batch(preds[:, -1], targets[:, -1]))
    model.reset_state()
    return float(np.mean(scores)) if scores else 0.0


best_ssim  = 0.0
global_step = 0
history     = {'loss': [], 'ssim': []}

print(f'Training for {CFG["n_epochs"]} epochs...')
t_start = time.time()

for epoch in range(CFG['n_epochs']):
    model.train()
    epoch_loss = []

    for batch_idx, (voxels, targets) in enumerate(train_dl):
        voxels  = voxels.to(device)    # (B, T, C, H, W)
        targets = targets.to(device)   # (B, T, 1, H, W)
        B_b, T, C, H, W = voxels.shape

        model.reset_state()
        preds = []
        for t in range(T):
            preds.append(model(voxels[:, t]))
        preds = torch.stack(preds, dim=1)  # (B, T, 1, H, W)

        loss = loss_fn(
            preds.view(-1, 1, H, W),
            targets.view(-1, 1, H, W)
        )

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optim.step()

        epoch_loss.append(loss.item())
        global_step += 1

        # Checkpoint every N steps
        if global_step % CFG['ckpt_every'] == 0:
            ckpt_path = CKPT_DIR / f'ckpt_step{global_step:07d}.pt'
            torch.save({
                'epoch': epoch, 'step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optim.state_dict(),
                'loss': loss.item(),
            }, ckpt_path)

    mean_loss = float(np.mean(epoch_loss))
    mean_ssim = evaluate(val_dl)   # held-out validation SSIM
    history['loss'].append(mean_loss)
    history['ssim'].append(mean_ssim)

    sched.step()

    elapsed = (time.time() - t_start) / 60
    print(f'Epoch {epoch+1:3d}/{CFG["n_epochs"]} | '
          f'loss={mean_loss:.4f} | val_ssim={mean_ssim:.4f} | '
          f'{elapsed:.1f} min elapsed')

    # Save best model
    if mean_ssim > best_ssim:
        best_ssim = mean_ssim
        best_path = Path(CFG['drive_dir']) / 'decoder_v1.pt'
        torch.save({
            'epoch': epoch,
            'ssim': best_ssim,
            'config': {k: v for k, v in CFG.items() if k != 'device'},
            'model_state_dict': model.state_dict(),
        }, best_path)
        print(f'  * New best val SSIM={best_ssim:.4f} — saved to {best_path}')

print(f'\nTraining complete. Best SSIM: {best_ssim:.4f}')
print(f'Model saved to: {CFG["drive_dir"]}/decoder_v1.pt')

In [ ]:
# ── 8. Plot training curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['loss'], color='#1A56DB', linewidth=2)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history['ssim'], color='#16A34A', linewidth=2)
ax2.set_title('Validation SSIM'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('SSIM')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
curve_path = Path(CFG['drive_dir']) / 'training_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {curve_path}')

In [ ]:
# ── 9. Export to ONNX ───────────────────────────────────────────────────
# Exports a single recurrent step: inputs voxel + ConvLSTM states, outputs frame + new states.
# For a numerical check against PyTorch run eval/export_onnx.py locally.
from neurocodec.decoder import export_onnx

onnx_path = Path(CFG['drive_dir']) / 'decoder.onnx'
inputs = export_onnx(model, onnx_path, height=CFG['img_h'], width=CFG['img_w'])
print(f'ONNX model exported to {onnx_path}\nInputs: {inputs}')
